# Bagging Classifiers (In-Depth)

## 1. Visual Intuitive Analysis: Single Estimator vs. Bagging Ensemble

Bagging Classifiers help reduce the variance of high-variance models by training multiple estimators on different bootstrap samples and aggregating their predictions.

### Single Decision Tree (Unconstrained)

- Suffers from **overfitting**.
- Creates highly complex and irregular decision boundaries.
- Attempts to fit every small variation and noise point in the training data.
- Usually achieves very high training accuracy but struggles to generalize on unseen data.
- Represents a **low-bias, high-variance** model.

### Bagging Classifier (Ensemble of Decision Trees)

- Produces a smoother and more generalized decision boundary.
- Reduces the impact of noise and outliers.
- Improves generalization performance on unseen data.
- Maintains the low bias of individual trees while significantly reducing variance.
- Represents a more stable and robust predictive model.

---

## 2. Bagging Architecture

### Overall Workflow

```text
                    Master Training Dataset (D)
                                 |
                                 |
                     Bootstrap Sampling
                                 |
      -------------------------------------------------
      |                 |                 |           |
      v                 v                 v           v

   Dataset D₁       Dataset D₂       Dataset D₃    Dataset Dₙ
      |                 |                 |           |
      v                 v                 v           v

 Decision Tree 1  Decision Tree 2  Decision Tree 3  Decision Tree n
      |                 |                 |           |
      -------------------------------------------------
                                 |
                                 |
                           Aggregation
                                 |
        ------------------------------------------------
        |                                              |
        v                                              v

 Classification                              Regression
 Majority Voting                          Average Prediction
                                 |
                                 |
                          Final Prediction
```

### Key Components

#### Bootstrapping

- Random subsets are generated from the original dataset.
- Sampling is usually performed **with replacement**.
- Each estimator receives a slightly different version of the training data.

#### Homogeneous Base Estimators

- All base models are typically the same algorithm.
- Most commonly, Decision Trees are used.
- Diversity comes from different training samples rather than different algorithms.

#### Aggregation

**Classification Tasks**

- Every estimator predicts a class label.
- Final prediction is obtained using **Majority Voting (Mode)**.

**Regression Tasks**

- Every estimator predicts a numerical value.
- Final prediction is obtained using the **Mean (Average)** of all predictions.

---

## 3. Why Bagging Works: The Bias-Variance Tradeoff

### The Problem

Machine Learning models generally suffer from two types of errors:

#### Bias

- Error due to overly simplistic assumptions.
- Leads to underfitting.

#### Variance

- Error caused by sensitivity to training data fluctuations.
- Leads to overfitting.

High-capacity models such as fully-grown Decision Trees generally have:

- Low Bias
- High Variance

### The Solution: Bagging

Bagging primarily reduces **Variance** while preserving the low bias of the base estimators.

#### How Variance Reduction Happens

1. Each model sees a different bootstrap sample.
2. Outliers affect only a subset of models.
3. Individual overfitting patterns differ across estimators.
4. Aggregation averages out these fluctuations.

### Result

$$
\text{Low Bias} + \text{Reduced Variance}
=
\text{Better Generalization}
$$

---

## 4. Types of Bagging

Different combinations of row sampling and column sampling create four major variants.

### Architecture Diagram

```text
                    [ Bagging Variations ]
                               |
        -------------------------------------------------
        |                                               |
        v                                               v

     Row Sampling                               Column Sampling
                                                   Involved
        |                                               |
   --------------                               ----------------
   |            |                               |              |
   v            v                               v              v

Bagging      Pasting                    Random Subspaces   Random Patches
```

---

### A. Standard Bagging

#### Configuration

- Row Sampling: With Replacement
- Column Sampling: Disabled

#### Characteristics

- Most common form of Bagging.
- Creates high diversity among estimators.
- Excellent for reducing variance.

---

### B. Pasting

#### Configuration

- Row Sampling: Without Replacement
- Column Sampling: Disabled

#### Characteristics

- No duplicate rows appear within subsets.
- Often used when dataset size is large.

---

### C. Random Subspaces

#### Configuration

- Row Sampling: Disabled
- Column Sampling: Enabled

#### Characteristics

- Every estimator sees all rows.
- Each estimator sees only a subset of features.
- Useful for high-dimensional datasets.

#### Applications

- Text Classification
- NLP
- Image Processing

---

### D. Random Patches

#### Configuration

- Row Sampling: Enabled
- Column Sampling: Enabled

#### Characteristics

- Samples both rows and columns.
- Maximizes diversity among estimators.
- Particularly useful for very large datasets.

---

## 5. Out-of-Bag (OOB) Validation

### What Are OOB Samples?

When bootstrap sampling is performed with replacement:

- Some samples are selected multiple times.
- Some samples are never selected.

The samples not selected for a particular estimator are called **Out-of-Bag (OOB) Samples**.

### Mathematical Insight

As dataset size becomes large:

$$
\lim_{N \to \infty}
\left(1-\frac{1}{N}\right)^N
=
\frac{1}{e}
\approx 0.368
$$

Therefore:

$$
36.8\%
$$

of training samples are expected to remain unseen by a given estimator.

### Why OOB Is Useful

These unseen samples can act as an internal validation set.

Benefits:

- No separate validation split required.
- Efficient use of data.
- Fast performance estimation.

---

## 6. Engineering Best Practices

### Bagging vs Pasting

- Bagging generally performs better than Pasting.
- Sampling with replacement introduces greater estimator diversity.

### Choosing `max_samples`

A practical range is:

$$
0.25 \leq \text{max\_samples} \leq 0.50
$$

Benefits:

- Faster training
- Lower computational cost
- Strong ensemble performance

### Feature Sampling

Feature sampling is useful when:

- Dataset contains hundreds or thousands of features.
- High-dimensional feature spaces exist.

Examples:

- NLP datasets
- Image datasets
- Genomics datasets

For low-dimensional datasets, aggressive feature sampling can hurt performance.

### Choosing Base Estimators

Bagging works best with:

- Decision Trees
- Other high-variance models

Decision Trees are particularly effective because:

- They have low bias.
- They are highly sensitive to training data changes.
- Bagging effectively stabilizes their predictions.

---

## 7. Relationship Between Bagging and Random Forest

A Random Forest can be viewed as a specialized extension of Bagging.

### Bagging

- Bootstrap row sampling
- Decision Trees
- All features available at every split

### Random Forest

- Bootstrap row sampling
- Decision Trees
- Random subset of features considered at each split

This additional feature randomness increases diversity and usually improves performance further.

---

## 8. Key Takeaways

- Bagging stands for **Bootstrap Aggregating**.
- It is designed primarily to reduce **Variance**.
- Works best with high-variance models such as Decision Trees.
- Multiple bootstrap datasets create estimator diversity.
- Predictions are combined using:
  - Majority Voting for Classification
  - Mean Averaging for Regression
- Four major variants exist:
  - Bagging
  - Pasting
  - Random Subspaces
  - Random Patches
- Out-of-Bag samples provide built-in validation.
- Random Forest is a specialized and highly optimized form of Bagging.

In [1]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

In [2]:
x,y = make_classification(n_samples=10000, n_features=10,n_informative=3)

In [3]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [4]:
# Normal DT 

dt = DecisionTreeClassifier(random_state=42)
dt.fit(x_train,y_train)
y_pred = dt.predict(x_test)

print("Decision Tree accuracy",accuracy_score(y_test,y_pred))

Decision Tree accuracy 0.904


# Bagging

In [5]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),  # Base estimator (each model is a Decision Tree)
    n_estimators=500,                    # Total number of Decision Trees in the ensemble
    max_samples=0.5,                     # Each tree gets 50% of the training rows
    bootstrap=True,                      # Sample rows with replacement (Standard Bagging)
    random_state=42                      # Ensures reproducible results
)

In [6]:
bag.fit(x_train,y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeClassifier()
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",500
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",0.5
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [7]:
y_pred = bag.predict(x_test)

In [8]:
accuracy_score(y_test,y_pred)

0.943

In [9]:
# Rows used in bagging 
bag.estimators_samples_[0].shape

(4000,)

In [10]:
# cols
bag.estimators_features_[0].shape

(10,)

# Bagging using SVM

In [12]:
bag = BaggingClassifier(
    estimator=SVC(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42
)

In [13]:
bag.fit(x_train,y_train)
y_pred = bag.predict(x_test)
print("Bagging using SVM",accuracy_score(y_test,y_pred))

Bagging using SVM 0.9055


# Pasting

In [14]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),  # Base model used in the ensemble (Decision Tree)
    n_estimators=500,                         # Total number of Decision Trees to train
    max_samples=0.25,                         # Each tree gets 25% of the training rows
    bootstrap=False,                          # False = Pasting (sampling without replacement)
    random_state=42,                          # Ensures reproducible results
    verbose=1,                                # Displays training progress in the console
    n_jobs=-1                                 # Uses all available CPU cores for parallel training
)

In [15]:
bag.fit(x_train,y_train)
y_pred = bag.predict(x_test)
print("Pasting classifier",accuracy_score(y_test,y_pred))

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   2 out of  16 | elapsed:   19.5s remaining:  2.3min
[Parallel(n_jobs=16)]: Done  16 out of  16 | elapsed:   21.2s finished
[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done   2 out of  16 | elapsed:    0.1s remaining:    1.1s


Pasting classifier 0.944


[Parallel(n_jobs=16)]: Done  16 out of  16 | elapsed:    0.5s finished


# Random Subspaces

In [17]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),  # Base model (Decision Tree)
    n_estimators=500,                         # Total number of Decision Trees
    max_samples=1.0,                          # Each tree gets 100% of training rows
    bootstrap=False,                          # Sampling without replacement (Pasting)
    max_features=0.5,                         # Each tree gets 50% of the feature columns
    bootstrap_features=True,                  # Sample features with replacement
    random_state=42                           # Reproducible results
)

In [18]:
bag.fit(x_train,y_train)
y_pred = bag.predict(x_test)
print("Random Subspaces classifier",accuracy_score(y_test,y_pred))

Random Subspaces classifier 0.935


In [19]:
bag.estimators_samples_[0].shape

(8000,)

In [20]:
bag.estimators_features_[0].shape

(5,)

# Random Patches 

In [22]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),  # Base estimator (Decision Tree)
    n_estimators=500,                         # Total number of trees in the ensemble
    max_samples=0.25,                         # Each tree trains on 25% of the training rows
    bootstrap=True,                           # Row sampling with replacement
    max_features=0.5,                         # Each tree uses 50% of the available features
    bootstrap_features=True,                  # Feature sampling with replacement
    random_state=42                           # Ensures reproducible results
)

In [23]:
bag.fit(x_train,y_train)
y_pred = bag.predict(x_test)
print("Random Patches classifier",accuracy_score(y_test,y_pred))

Random Patches classifier 0.9305


### Memory Trick 
Only Rows Sampled      → Bagging / Pasting  
Only Columns Sampled   → Random Subspaces  
Rows + Columns Sampled → Random Patches  

# OOB Score  

In [25]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),  # Base estimator (Decision Tree)
    n_estimators=500,                         # Total number of trees in the ensemble
    max_samples=0.25,                         # Each tree trains on 25% of the training rows
    bootstrap=True,                           # Sampling with replacement (Standard Bagging)
    oob_score=True,                           # Calculate Out-of-Bag score using unseen samples
    random_state=42                           # Ensures reproducible results
)

In [26]:
bag.fit(x_train,y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeClassifier()
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",500
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",0.25
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",True
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [27]:
# Out-of-Bag (OOB) Accuracy
# Uses the ~36.8% samples not selected during bootstrap sampling
# Acts like an internal validation score
print("OOB Score:", bag.oob_score_)

OOB Score: 0.947125


In [28]:
y_pred = bag.predict(x_test)
print("Accuracy",accuracy_score(y_test,y_pred))

Accuracy 0.9415


# Bagging Tips
- Bagging generally gives better results than Pasting  
- Good results come around the 25% to 50% row sampling mark  
- Random patches and subspaces should be used while dealing with high dimensional data  
- To find the correct hyperparameter values we can do GridSearchCV/RandomSearchCV  

# Applying GridSearchCV

In [29]:
from sklearn.model_selection import GridSearchCV

In [30]:
parameters = {
    'n_estimators': [50,100,500], 
    'max_samples': [0.1,0.4,0.7,1.0],
    'bootstrap' : [True,False],
    'max_features' : [0.1,0.4,0.7,1.0]
    }

In [31]:
search = GridSearchCV(BaggingClassifier(), parameters, cv=5)

In [ ]:
search.fit(x_train,y_train)

In [ ]:
search.best_params_

In [ ]:
search.best_score_